## Creating a spark session

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('MySpark') \
    .getOrCreate()

## Load data from CSV

In [0]:
df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('sep', ',') \
    .csv('/Workspace/Users/vanshkrjain@gmail.com/Sample - Superstore.csv')

### First few rows

In [0]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

### Column names

In [0]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

### Data types

In [0]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Data cleaning

### Removing Duplicates

In [0]:
df.dropDuplicates().count()

9994

### Drop null values

In [0]:
df.na.drop().count()

9994

### Fill missing values

In [0]:
df.na.fill(0).show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

### Handling inconsistent data

In [0]:
df.describe().show()

+-------+------------------+--------------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|summary|            Row ID|      Order ID|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|    City|  State|       Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|             Sales|          Quantity|          Discount|            Profit|
+-------+------------------+--------------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|  count|              9994|          9994|          9994|       9994|              9994|       9994|        

In [0]:
df.select(['Country', 'City', 'State']).distinct().show()

+-------------+--------------+--------------+
|      Country|          City|         State|
+-------------+--------------+--------------+
|United States|  Philadelphia|  Pennsylvania|
|United States|        Durham|North Carolina|
|United States|     Urbandale|          Iowa|
|United States|     Roseville|    California|
|United States|        Dallas|         Texas|
|United States|      Lakewood|    New Jersey|
|United States|        Layton|          Utah|
|United States|    Wilmington|North Carolina|
|United States|      Gastonia|North Carolina|
|United States|    Costa Mesa|    California|
|United States|    Marysville|    Washington|
|United States|        Aurora|      Illinois|
|United States|        Linden|    New Jersey|
|United States|        Toledo|          Ohio|
|United States|    Farmington|    New Mexico|
|United States|Virginia Beach|      Virginia|
|United States|  Jacksonville|North Carolina|
|United States|       Clinton|      Maryland|
|United States|  Cedar Rapids|    

In [0]:
from pyspark.sql import functions as f
df.filter(f.col('Ship Date') < f.col('Order Date')).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [0]:
df.filter(~f.col('Postal Code').rlike('^[0-9]{5}$')).show()

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+----------+-------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|      City|        State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+----------+-------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|   186|CA-2016-105018|2016-11-28|2016-12-02|Standard Class|   SK-19990|   Sally Knutson|   Consumer|United States| Fairfield|  Connecticut|       6824|  East|OFF-BI-10001890|Office Supplies|     Binders|Avery P

In [0]:
from pyspark.sql.functions import expr

df = (df
      .withColumn("Sales", expr("try_cast(Sales as double)"))
      .withColumn("Quantity", expr("try_cast(Quantity as double)"))
      .withColumn("Discount", expr("try_cast(Discount as double)"))
)

In [0]:
df.select("Sales", "Quantity", "Discount").show(5)
df.printSchema()

+--------+--------+--------+
|   Sales|Quantity|Discount|
+--------+--------+--------+
|  261.96|     2.0|     0.0|
|  731.94|     3.0|     0.0|
|   14.62|     2.0|     0.0|
|957.5775|     5.0|    0.45|
|  22.368|     2.0|     0.2|
+--------+--------+--------+
only showing top 5 rows
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (

### Filter data

In [0]:
df.select(['Category', 'Region', 'Quantity']).distinct().show()

+---------------+-------+--------+
|       Category| Region|Quantity|
+---------------+-------+--------+
|Office Supplies|  South|     3.0|
|     Technology|Central|     7.0|
|      Furniture|Central|     1.0|
|      Furniture|   West|     2.0|
|     Technology|   West|    13.0|
|Office Supplies|  South|     5.0|
|      Furniture|   East|     8.0|
|     Technology|   West|    10.0|
|Office Supplies|   West|  35.352|
|      Furniture|   East|    13.0|
|      Furniture|   West|    NULL|
|Office Supplies|   East|  122.94|
|Office Supplies|   West|   31.32|
|Office Supplies|   West|  24.784|
|Office Supplies|   West|    71.6|
|      Furniture|  South|    NULL|
|Office Supplies|  South|    12.0|
|Office Supplies|Central| 123.552|
|Office Supplies|   West|  98.352|
|Office Supplies|  South|  86.058|
+---------------+-------+--------+
only showing top 20 rows


In [0]:
df_category = df.filter(f.col('Category') == 'Furniture')
df_category.show()

+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|    Segment|      Country|           City|       State|Postal Code| Region|     Product ID| Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------+------------+--------------------+--------+--------+--------+----------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute|   Consumer|United States|      Henderson|    Kentucky|      42420|  South|FUR-BO-10001798|Furniture|   Bookcases|Bush Somerse

In [0]:
df_region = df.filter(f.col('Region') == 'Central')
df_region.show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+----------+---------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|      City|    State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+----------+---------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|    15|US-2015-118983|2015-11-22|2015-11-26|Standard Class|   HP-14815|    Harold Pawlan|Home Office|United States|Fort Worth|    Texas|      76106|Central|OFF-AP-10002311|Office Supplies|  Appliances|Holmes Replacemen.

## Transform Data

In [0]:
df_renamed = df.withColumnRenamed('Postal Code', 'Pin Code').withColumnRenamed('Region', 'Landmark')

df_renamed.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+--------+--------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Pin Code|Landmark|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+--------+--------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|   42420|   South|FUR-BO-10001798|      Furni

## Aggregation

In [0]:
total_rows = df.count()
print(f'Total Rows: {total_rows}')

Total Rows: 9994


In [0]:
aggregation = df.agg(
    f.min('Discount').alias('minimum_discount'),
    f.max('Discount').alias('maximum_discount')
)
aggregation.show()

+----------------+----------------+
|minimum_discount|maximum_discount|
+----------------+----------------+
|             0.0|         295.056|
+----------------+----------------+



In [0]:
agg = df.agg(
    f.avg('Discount').alias('average discount')
)
agg.show()

+------------------+
|  average discount|
+------------------+
|0.3155949113493005|
+------------------+



## Group Data

In [0]:
df.groupBy("Category").count().show()

+---------------+-----+
|       Category|count|
+---------------+-----+
|Office Supplies| 6026|
|     Technology| 1847|
|      Furniture| 2121|
+---------------+-----+



In [0]:
df.groupBy("Category").sum("Sales").show()

+---------------+-----------------+
|       Category|       sum(Sales)|
+---------------+-----------------+
|Office Supplies|703502.9280000009|
|     Technology|835900.0669999991|
|      Furniture|733046.8613000009|
+---------------+-----------------+



In [0]:
df.groupBy("Category").avg('Sales').show()

+---------------+------------------+
|       Category|        avg(Sales)|
+---------------+------------------+
|Office Supplies|121.69225531914908|
|     Technology| 454.5405475802061|
|      Furniture|353.44593119575745|
+---------------+------------------+



In [0]:
pdf = df.toPandas()

In [0]:
pdf.to_csv('/tmp/result.csv', index = False)

In [0]:
import os
os.listdir('/tmp')

['hsperfdata_root',
 'databricks-jni13851322723983726785',
 'dbr-consolidated-secret-conf-envvars',
 'matplotlib-root',
 'dbr.conf',
 'pre_start.sh',
 'driver-env.sh',
 'keyutil_spark.host.local_13912181161209006851.crt',
 'backup_bind_mount',
 'validate_container_security.py',
 'result.csv',
 'clean_up_local_disk.sh',
 'prefetch_critical_files.sh',
 'chauffeur-env.sh',
 'custom-spark.conf',
 'tmp6pr9j7eb',
 'temp10794792980324642345snapstart-jni.so',
 'start_pre_checkpoint',
 'chauffeur-daemon.pid',
 'chauffeur-daemon-params',
 'keyutil_spark.host.local_16232400608960326134.key',
 'driver-daemon.pid',
 'temp3752921607746991231snapstart-jni.so',
 'master-params',
 'dbr_entry_point.py',
 'driver-daemon-params',
 'backup_process_log',
 'python_lsp_logs']

In [0]:
import shutil

shutil.copy("/tmp/result.csv",
            "/Workspace/Users/vanshkrjain@gmail.com/result.csv")

'/Workspace/Users/vanshkrjain@gmail.com/result.csv'